### Sensor data analysis workflow - EU REED

#### 1) Import packages and set filepath

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tarfile
from pathlib import Path
from reed import utils
from collections import defaultdict, Counter
from functools import reduce
import re

In [12]:
config = utils.Config()
#test

In [13]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"/"Data"
tool_descriptions_path = config.REED_PATH/'Data'/"MonitoringData"

#### 2) Data Import

First, the sensor data is imported. Next, the additional information required for the mapping is imported.

In [14]:
def parse_event_and_type_from_csv_filename(csv_filename: str):
    """
    CSV-filnavn (uden .csv) ser ud som fx:
      OF10001_G_TXG_REGRHM_92SQBY
    measurement_type = sidste 6 tegn: 92SQBY
    event_id = resten (uden evt. trailing underscore)
    """
    base = Path(csv_filename).name  # fjerner evt. sti inde i tar
    stem = base[:-4] if base.lower().endswith(".csv") else base

    if len(stem) < 6:
        return stem, None

    measurement_type = stem[-6:]
    event_part = stem[:-6]
    event_id = event_part[:-1] if event_part.endswith("_") else event_part
    return event_id, measurement_type


all_records = []

for company_folder in monitoring_data_path.iterdir():
    if not company_folder.is_dir():
        continue

    # Firmnavn = første del før " - "
    parts = [p.strip() for p in company_folder.name.split(" - ")]
    if len(parts) < 1:
        print("Advarsel – unexpected name of folder:", company_folder.name)
        continue

    company = parts[0].strip().lower()
    maskinenavn = " - ".join(parts[1:]) if len(parts) > 1 else None  # valgfrit, men nyttigt metadata

    # Find ALLE tar-filer i denne topmappe (inkl. undermapper for robusthed)
    tar_files = list(company_folder.rglob("*.tar")) + list(company_folder.rglob("*.tar.gz"))

    if not tar_files:
        print(f"Ingen .tar/.tar.gz-filer i {company_folder.name}")
        continue

    # Til dublet-advarsel: csv basenames indenfor denne company_folder
    seen_csv_basenames = []

    for tar_path in tar_files:
        try:
            # r:* autodetekterer .tar vs .tar.gz
            with tarfile.open(tar_path, mode="r:*") as tf:
                csv_members = [m for m in tf.getmembers() if m.name.lower().endswith(".csv")]

                if not csv_members:
                    print(f"Ingen CSV fundet i {tar_path.name}")
                    continue

                for csv_member in csv_members:
                    csv_basename = Path(csv_member.name).name
                    seen_csv_basenames.append(csv_basename)

                    event_id, measurement_type = parse_event_and_type_from_csv_filename(csv_basename)

                    f = tf.extractfile(csv_member)
                    if f is None:
                        print(f"Advarsel – kunne ikke extracte {csv_member.name} i {tar_path}")
                        continue

                    df = pd.read_csv(f)

                    all_records.append({
                        "company": company,
                        "maskinenavn": maskinenavn,
                        "event_id": event_id,
                        "measurement_type": measurement_type,
                        "tar_path": tar_path,
                        "csv_name": csv_member.name,     # sti inde i tar
                        "csv_basename": csv_basename,    # kun filnavn
                        "data": df,
                    })

        except tarfile.TarError as e:
            print(f"Advarsel – kunne ikke åbne arkiv: {tar_path} | {e}")

    # Dublet-melding for ens CSV-filnavne (basenames) indenfor company_folder
    c = Counter(seen_csv_basenames)
    dups = [name for name, cnt in c.items() if cnt > 1]
    if dups:
        print(f"\n⚠️ Dublet CSV-filnavne fundet i '{company_folder.name}':")
        for name in sorted(dups):
            print(f"  - {name} (x{c[name]})")

print("\nTotal amount of CSV imports:", len(all_records))

# === group company and event ===
events = defaultdict(list)
for rec in all_records:
    events[(rec["company"], rec["event_id"])].append(rec)

print("Events:", len(events))


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\TimWeber\\OneDrive - 2.-0 LCA Consultants ApS\\Intranet - HorizonEurope\\EU_REED_1277\\Data\\MonitoringData\\Data'

In [ ]:
def load_tool_operation_keys(tool_descriptions_path: Path):
    """
    Læser tool_operation_combined.xlsx fra tool_descriptions_path og returnerer:
      - key_company_tools: kombineret DF fra arkene KOMATSU, WWR, GOIMEK
          kolonner: company, Tool_Number, OP TYPE
      - key_reed_tools: DF fra arket REED (originale kolonnenavne)
    """
    xlsx_path = tool_descriptions_path / "tool_operation_combined.xlsx"

    # Ark der skal kombineres
    company_sheets = ["KOMATSU", "WWR", "GOIMEK"]

    dfs = []
    for sheet in company_sheets:
        df = pd.read_excel(xlsx_path, sheet_name=sheet, skiprows=1)
        df_out = df[["TOOL", "OP TYPE"]].copy()
        df_out.insert(0, "company", sheet)
        df_out = df_out.rename(columns={"TOOL": "Tool_Number"})
        dfs.append(df_out)

        df_out["Tool_Number"] = (
            df_out["Tool_Number"]
            .astype(str)
            .str.replace(" ", "", regex=False)
            .str.split("/")
        )
        df_out = df_out.explode("Tool_Number", ignore_index=True)
        df_out["Tool_Number"] = pd.to_numeric(df_out["Tool_Number"], errors="coerce")

        dfs.append(df_out)

    key_company_tools = pd.concat(dfs, ignore_index=True)

    # REED som separat df med originale kolonnenavne
    key_reed_tools = pd.read_excel(xlsx_path, sheet_name="REED")

    return key_company_tools, key_reed_tools

key_company_tools, key_reed_tools = load_tool_operation_keys(tool_descriptions_path)
key_company_tools["Tool_Number"] = pd.to_numeric(key_company_tools["Tool_Number"], errors="coerce")
key_company_tools["company"] = key_company_tools["company"].str.strip().str.lower()




#### 3) Data analysis

The different required inputs are merged into one dataframe and grouped by the parameters that are interesting in this project. Namely, those are event, energy consumption, tool_number, reed tool (our LCA process) and operation time.

In [ ]:
def merge_records(records):
    """
    Creating one dataFrame for each event.

    """
    dfs = []
    cols = ["timestamp", "Tool_Number", "Power_Active", "Energy_Total"]

    merged = None
    
    for i, r in enumerate(records):
        df = r["data"].copy()
        df = df[[c for c in cols if c in df.columns]].copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.floor('s')
        df = df.set_index("timestamp")

        if merged is None:
            merged = df
        else:
            merged = merged.join(df, how="outer")

    merged = merged.sort_index()
            
    return merged

In [ ]:
all_grouped = []  
event_summaries = []
for (company, event_id), records in events.items():
    merged_df = merge_records(records)
    if merged_df.empty or "Power_Active" not in merged_df.columns:
        print("\n--- MISSING Power_Active ---")
        print("company:", company)
        print("event_id:", event_id)
        print("merged_df shape:", merged_df.shape)
        print("merged_df columns:", list(merged_df.columns))

        # extra: inspect what came in
        print("records type:", type(records), "len:", len(records) if hasattr(records, "__len__") else "n/a")
        if hasattr(records, "__len__") and len(records) > 0:
            r0 = records[0]
            print("first record type:", type(r0))
            if hasattr(r0, "columns"):
                print("first record columns:", list(r0.columns))
            elif isinstance(r0, dict):
                print("first record keys:", list(r0.keys()))
        # stop right here so you can inspect in VS Code
        breakpoint()

    merged_df["energy_consumption"] = merged_df["Power_Active"] / 3600

    merged_df = merged_df.reset_index()

    s = merged_df["Energy_Total"].dropna()

    if s.empty:
        meter_energy = np.nan
    else:
        meter_energy = s.iloc[-1] - s.iloc[0]

    event_summaries.append({
        "company": company,
        "event_id": event_id,
        "energy_meter_total": meter_energy,
    })

    def tool_hours(ts):
        ts = ts.sort_values()
        if ts.empty:
            return 0.0
        return (ts.iloc[-1] - ts.iloc[0]).total_seconds() / 3600.0

    # Group after tool_number
    grouped_df = (
        merged_df
        .groupby("Tool_Number", as_index=False)
        .agg(
            energy_consumption=("energy_consumption", "sum"),
            Power_Active=("Power_Active", "mean"),
            seconds=("timestamp", "count"),
            )
        )
    
    grouped_df["hours"] = grouped_df["seconds"] / 3600.0
    grouped_df["company"] = company
    grouped_df["event_id"] = event_id

    grouped_df = grouped_df.set_index(["company", "event_id", "Tool_Number"])

    all_grouped.append(grouped_df)

final_df = pd.concat(all_grouped)

# --- MERGE 1: tilføj OP TYPE via company + Tool_Number ---
final_df = final_df.reset_index()

final_df["Tool_Number"] = pd.to_numeric(final_df["Tool_Number"], errors="coerce")

final_df = final_df.merge(
    key_company_tools[["company", "Tool_Number", "OP TYPE"]],
    on=["company", "Tool_Number"],
    how="left"
)

# --- MERGE 2: tilføj REED-info via OP TYPE ---
final_df = final_df.merge(
    key_reed_tools,
    on="OP TYPE",
    how="left",
    suffixes=("", "_reed")
)

final_df["reed tool"] = final_df["reed tool"].fillna("machining, generic")
final_df = final_df.set_index(["company", "event_id", "Tool_Number"])


In [ ]:
print(final_df)

                                          energy_consumption  Power_Active  \
company event_id             Tool_Number                                     
goimek  OF10001_G_BQC_S8CF2G 0.0                    8.154403     11.789498   
                             1.0                   24.300319     11.436939   
                             1.0                   24.300319     11.436939   
                             1.0                   24.300319     11.436939   
                             1.0                   24.300319     11.436939   
...                                                      ...           ...   
wwr     OF10004_G_TWB_7SZ421 2694.0              4718.325664  13096.354965   
                             2695.0               959.320473   9700.993548   
                             2696.0              1756.356816  13036.875333   
                             2722.0               351.314922  10199.465472   
                             3350.0             28001.024411  13

The next step shows the tool numbers that have a missing match.

In [ ]:
missing_op_type_tools = (
    final_df
    .loc[final_df["OP TYPE"].isna()]
    .reset_index()[["company", "Tool_Number"]]
    .drop_duplicates()
    .sort_values(["company", "Tool_Number"])
    .reset_index(drop=True)
)

missing_op_type_tools



,company,Tool_Number
0,goimek,0.0
1,goimek,36.0
2,goimek,300.0
3,komatsu,0.0
4,komatsu,1.0
...,...,...
60,wwr,3121.0
61,wwr,3124.0
62,wwr,3135.0
63,wwr,3270.0


In order to look at single events, we now split the final_df to what we want to look at.

In [ ]:
def split_final_df_by_company_event(final_df):
    dfs = {}

    for (company, event_id), df in final_df.groupby(level=["company", "event_id"]):
        dfs[(company, event_id)] = df.reset_index()

    return dfs

event_dfs = split_final_df_by_company_event(final_df)

# eksempel
print(event_dfs[("komatsu", "OF10006_G_SLG_E8AJ1C")])


    company              event_id  Tool_Number  energy_consumption  \
0   komatsu  OF10006_G_SLG_E8AJ1C          0.0           22.901166   
1   komatsu  OF10006_G_SLG_E8AJ1C          1.0           18.325188   
2   komatsu  OF10006_G_SLG_E8AJ1C          2.0            7.289717   
3   komatsu  OF10006_G_SLG_E8AJ1C          3.0            2.889369   
4   komatsu  OF10006_G_SLG_E8AJ1C          3.0            2.889369   
5   komatsu  OF10006_G_SLG_E8AJ1C          3.0            2.889369   
6   komatsu  OF10006_G_SLG_E8AJ1C          3.0            2.889369   
7   komatsu  OF10006_G_SLG_E8AJ1C          3.0            2.889369   
8   komatsu  OF10006_G_SLG_E8AJ1C          4.0            1.117578   
9   komatsu  OF10006_G_SLG_E8AJ1C         23.0            2.981584   
10  komatsu  OF10006_G_SLG_E8AJ1C         24.0            0.562837   
11  komatsu  OF10006_G_SLG_E8AJ1C         31.0            2.139567   
12  komatsu  OF10006_G_SLG_E8AJ1C         31.0            2.139567   
13  komatsu  OF10006

#### 4) Prepare for import to LCA module

Energy consumption is given in kWh

In [ ]:
def sum_by_reed_tool(
    df,
    group_cols=("company", "event_id", "reed tool"),
    cols_to_sum=("energy_consumption", "hours", "seconds")
):
    return (
        df
        .drop(columns=["Tool_Number"], errors="ignore")
        .groupby(list(group_cols), as_index=False)
        .agg({col: "sum" for col in cols_to_sum})
    )

In [ ]:
energy_consumption_by_LCA_process = sum_by_reed_tool(
    event_dfs[("wwr", "OF10004_G_TWB_7SZ421")])

In [ ]:
print(energy_consumption_by_LCA_process)

  company              event_id           reed tool  energy_consumption  \
0     wwr  OF10004_G_TWB_7SZ421            drilling        54469.954977   
1     wwr  OF10004_G_TWB_7SZ421  machining, generic       893136.653766   
2     wwr  OF10004_G_TWB_7SZ421             milling       921071.398024   

        hours  seconds  
0    4.410833    15879  
1  104.086111   374710  
2   87.208333   313950  


### Additional information

Here the total energy consumption per event is calculated.

In [ ]:
df_totals = (
    final_df
    .reset_index()  # sikrer at company, event_id og Tool_Number er kolonner
    .groupby(["company", "event_id"], as_index=False)
    .agg({
        "energy_consumption": "sum",   
        "Power_Active": "mean",
        "hours": "sum"        
    })
    .set_index(["company", "event_id"])  # hvis du vil have dem som index igen
)

df_meter = (
    pd.DataFrame(event_summaries)
    .set_index(["company", "event_id"])
)

df_totals = df_totals.join(df_meter)

df_totals["rel_diff_calc_vs_meter"] = (
    (df_totals["energy_consumption"] - df_totals["energy_meter_total"])
    / df_totals["energy_meter_total"]
)




In [ ]:
print(df_totals)

                              energy_consumption  Power_Active       hours  \
company event_id                                                             
goimek  OF10001_G_BQC_S8CF2G        5.465944e+02     13.382317   43.932500   
        OF10002_G_BQC_S8CF2G        3.919829e+02     13.412874   30.228889   
        OF10003_G_BQC_S8CF2G        4.080839e+02     13.496392   30.879444   
        OF10004_G_BQC_S8CF2G        3.528260e+02     13.890126   26.036944   
komatsu OF10001_G_SLG_E8AJ1C        1.267761e+05   4892.112159   57.336944   
        OF10001_G_TXG_REGRHM        2.989877e+05   6252.574181   56.781944   
        OF10002_G_SLG_E8AJ1C        1.909440e+05   4828.529072   89.623611   
        OF10002_G_TXG_REGRHM        3.709700e+05   5966.770495  107.413333   
        OF10003_G_SLG_E8AJ1C        2.396522e+05   5264.222772   53.234167   
        OF10003_G_TXG_REGRHM        3.218350e+05   7054.271127   52.087222   
        OF10004_G_SLG_E8AJ1C        2.215201e+05   5389.803641  